In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Approximate Optimization Algorithm

*Nutzungsschätzung: 22 Minute op enem Heron r3 Prozessor (ACHTUNG: Dat es nur en Schätzung. Ding Laufzick kann variiere.)*
## Hintergrund
Dat Tutorial zeich wie mer der **Quantum Approximate Optimization Algorithm (QAOA)** – en hybrid (quanten-klassisch) iterativ Method – em Kontext vun Qiskit-Patterns implementiere. Mer löse zuerst dat **Maximum-Cut** (oder **Max-Cut**) Problem för ene kleine Grafen un dann liere mer, wie mer dat op Utility-Skala ussföhre. All Hardware-Ussfö hrunge em Tutorial sollte innerhalb vum Zicklimit för der frei zogänglich Open Plan funktioniere.

Dat Max-Cut-Problem es en Optimierungsproblem, dat schwer ze löse es (genauer gesaat es et en *NP-hardet* Problem) un hätt en Reih vun verschiedene Anwendunge em Clustering, Netzwerkwissenschaff un statistischer Physik. Dat Tutorial betrach ene Grafen vun Knoten, die durch Kante verbunde sin, un will de Knoten in zwei Menge opteile, esu dat de Anzahl vun de durch dä Schnitt durchquerte Kante maximiert weed.

![Illustration of a max-cut problem](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## Vorraussetzunge
Bevörr mer met däm Tutorial aanfange, kümmert Üch dorömm dat Ihr Folgendes installiert hätt:
- Qiskit SDK v1.0 oder später, met [Visualisierungs](https://docs.quantum.ibm.com/api/qiskit/visualization)-Ungerstützung
- Qiskit Runtime v0.22 oder später (`pip install qiskit-ibm-runtime`)

Zusätzlich bruch mer Zogang zo ener Instanz op dr [IBM Quantum Platform](/guides/cloud-setup). Pass op, dat dat Tutorial nit em Open Plan ussjeföhrt weede kann, weil et Workloads met [Sessions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session) ussföhrt, die nor met Premium Plan-Zogang verfügbar sin.
## Setup

In [1]:
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Deil I. QAOA em kleine Moßstab
Dä erste Deil vun däm Tutorial nömmt en klein Max-Cut-Problem, öm de Schrette fürr de Lösung vun enem Optimierungsproblem met enem Quantencomputer ze veranschauliche.

Öm en bessere Kontext ze jonn, bevörr mer dat Problem op ene Quantenalgorithmus afbilde, künnt Ihr besser verstehn, wie dat Max-Cut-Problem zo enem klassische kombinatorische Optimierungsproblem weed, indem mer zuerst de Minimierung vun ener Funktion $f(x)$ betrach

$$
\min_{x\in {0, 1}^n}f(x),
$$

wobei de Eingab $x$ ene Vektor es, wo sing Komponente jedem Knoten vun enem Grafen entspreche. Dann weed jede vun dene Komponente op entweder $0$ oder $1$ beschränk (wat representeet, of se em Schnitt enthalte sin oder nit). Dä kleinskaalich Beispielfall nemmp ene Grafen met $n=5$ Knoten.

Mer künnt en Funktion vun enem Knotepaar $i,j$ schriive, wat anze ich, of de entsprechend Kant $(i,j)$ em Schnitt litt. Zom Beispiel es de Funktion $x_i + x_j - 2 x_i x_j$ jenau dann 1, wenn entweder $x_i$ oder $x_j$ jlich 1 es (wat bedütt, dat de Kant em Schnitt litt) un sons null. Dat Problem, de Kante em Schnitt ze maximiere, kann formuleet weede als

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

wat als Minimierung ömjeschrevve weede kann in dä Form

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

Dat Minimum vun $f(x)$ in däm Fall litt vörr, wenn de Anzahl vun de durch dä Schnitt durchquerte Kante maximal es. Wie mer sieht, hätt dat noch nix met Quantencomputing ze donn. Mer muss dat Problem in jet ömformuliere, wat ene Quantencomputer verstohn kann.
Initialiseet Eur Problem, indem mer ene Grafen met $n=5$ Knoten erstelle.

In [2]:
n_small = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n_small, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### Schritt 1: Klassisch Eingab op en Quantenproblem afbilde
Dä erste Schritt vum Pattern besteit dorin, dat klassisch Problem (Graf) op quantenmechanisch **Schaltkreise** un **Operatoren** afzebilde. Dozö sin drei Hauptschrette ze unge rnämme:

1. Verwendung vun ener Reih vun mathematische Ömformulierunge, öm dat Problem mithilf vun dä Notation vun Quadratic Unconstrained Binary Optimization (QUBO) Probleme darzestelle.
2. Ömformulierung vum Optimierungsproblem als Hamilton-Operator, för dä dr Grundzostand dä Lösung entsprich, wo de Kostenfunktion minimeet.
3. Erstellung vun enem Quantenschaltkreis, dä dr Grundzostand vun däm Hamilton-Operator övver ene Prozess ähnlich däm Quantum Annealing vörbereidt.

**Pass op:** En dä QAOA-Methodik wolle mer letztendlich ene Operator (**Hamilton-Operator**) han, dä de **Kostenfunktion** vun unsere hybride Algorithmus darstellt, suwie ene parametrisierte Schaltkreis (**Ansatz**), dä Quantenzuständ met Kandidatelösunge för dat Problem darstellt. Mer künne us dene Kandidatezuständ sample un se dann met dä Kostenfunktion bewerte.

#### Graf &rarr; Optimierungsproblem
Dä erste Schritt vun dä Afbildung es en Notationsänderung. Dat Folgende drück dat Problem in QUBO-Notation us:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

wobei $Q$ en $n\times n$ Matrix vun reelle Zahle es, $n$ dä Anzahl vun de Knoten in Eurem Grafen entsprich, $x$ dä ovve eijeföhrte Vektor vun binäre Variable es un $x^T$ de Transponiert vum Vektor $x$ bezeichnet.

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### Optimierungsproblem &rarr; Hamilton-Operator
Mer künne dann dat QUBO-Problem als **Hamilton-Operator** ömformuliere (hee en Matrix, wo de Energie vun enem System darstellt):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**Ömformulierungsschrette vum QAOA-Problem zum Hamilton-Operator**
</summary>

Öm ze zeiche, wie dat QAOA-Problem op die Wiis ömjeschrevve weede kann, ersetze mer zuerst de binär Variable $x_i$ durch ene neue Satz vun Variable $z_i\in{-1, 1}$ övver

$$
x_i = \frac{1-z_i}{2}.
$$

Hee künnt mer sinn, dat wenn $x_i$ jlich $0$ es, dann $z_i$ jlich $1$ sin muss. Wenn de $x_i$ durch de $z_i$ em Optimierungsproblem ($x^TQx$) ersetzt weede, kann en äquivalent Formulierung erhalte weede.

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

Wenn mer jetz $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$ definiere, dä Vörfaktor entferne un dä konstant Term $n^2$ weglosse, erhalte mer de beid äquivalent Formulierunge vum selve Optimierungsproblem.

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

Hee hänk $b$ vun $Q$ av. Pass op, dat mer för d'Erlangung vun $z^TQz + b^Tz$ dä Faktor 1/4 un ene konstant Offset vun $n^2$ wechjelosse han, wo bei dä Optimierung kein Roll spille.

Öm jetz en Quantenformulierung vum Problem ze erhalte, erhevve mer de Variable $z_i$ zo ener Pauli $Z$ Matrix, wie ener $2\times 2$ Matrix vun dä Form

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

Wenn mer die Matrize em ovve Optimierungsproblem eisetze, erhalte mer dä folgende Hamilton-Operator

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*Pass och op, dat de $Z$ Matrize em Rechenruum vum Quantencomputer eingebett sin, dat heiß in enem Hilbert-Ruum vun dä Jröß $2^n\times 2^n$. Deswäje sollte mer Terme wie $Z_iZ_j$ als dat Tensorprodukt $Z_i\otimes Z_j$ verstehn, wat em $2^n\times 2^n$ Hilbert-Ruum eingebett es. Zom Beispiel weed in enem Problem met fönf Entscheidungsvariable dä Term $Z_1Z_3$ als $I\otimes Z_3\otimes I\otimes Z_1\otimes I$ verstande, wobei $I$ de $2\times 2$ Einheitsmatrix es.*
</details>

Dä Hamilton-Operator weed als **Kostenfunktions-Hamilton-Operator** bezeichnet. Hä hätt de Eijenschaff, dat sing Grundzostand dä Lösung entsprich, wo de **Kostenfunktion $f(x)$ minimeet**.
Öm Eur Optimierungsproblem ze löse, müsse mer jetz dä Grundzostand vun $H_C$ (oder ene Zostand met huher Övverlappung domit) opem Quantencomputer präpariere. Dat Sample us däm Zostand weed dann met huher Wahrscheinlichkeit de Lösung zo $min~f(x)$ liefere.
Betrach mer jetz dä Hamilton-Operator $H_C$ för dat **Max-Cut** Problem. Jedem Knoten vum Grafen weed en Qubit em Zostand $|0\rangle$ oder $|1\rangle$ zojeordnet, wobei dä Wert de Meng anzich, zo dä dä Knoten jehürt. Dat Ziel vum Problem es et, de Anzahl vun de Kante $(v_1, v_2)$ ze maximiere, för wo $v_1 = |0\rangle$ un $v_2 = |1\rangle$ jilt, oder ömjekehrt. Wenn mer dä $Z$ Operator jedem Qubit zoordne, wobei

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

dann jehürt en Kant $(v_1, v_2)$ zom Schnitt, wenn dä Eigenwert vun $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$ es; met andere Wööt, de met $v_1$ un $v_2$ assoziierte Qubits sin ungerschiedlich. Esu jehürt $(v_1, v_2)$ nit zom Schnitt, wenn dä Eigenwert vun $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$ es. Pass op, dat uns dä jenau Qubit-Zostand, dä jedem Knoten zojeordnet es, nit interesseet, sons nor, of se övver en Kant henwäch jlich sin oder nit. Dat Max-Cut-Problem verlank vun uns, en Zoordnung vun de Qubits op de Knoten ze finge, esu dat dä Eigenwert vum folgende Hamilton-Operator minimeet weed
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

Met andere Wööt, $b_i = 0$ för all $i$ em Max-Cut-Problem. Dä Wert vun $Q_{ij}$ bezeichnet dat Jewicht vun dä Kant. In däm Tutorial betrach mer ene ungejewichtete Grafen, dat heiß $Q_{ij} = 1.0$ för all $i, j$.

In [3]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert graph edges to a list of ZZ Pauli terms.

    The returned list is in the sparse format expected by
    ``SparsePauliOp.from_sparse_list``: each element is
    ``(pauli_string, qubit_indices, coefficient)``.
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n_small)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Build the QAOA ansatz circuit

Use `QAOAAnsatz` to construct the parametrized QAOA circuit from the cost Hamiltonian. Here we use `reps=2` (two QAOA layers, four parameters: $\beta_0, \beta_1, \gamma_0, \gamma_1$).

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

Transpile the abstract circuit into hardware-native instructions. This step handles qubit mapping, gate decomposition, routing, and error suppression. See the transpilation [documentation](/docs/guides/transpile) for more information.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation. Level 3 is the most aggressive
# preset: slower to transpile, but produces shorter circuits that are
# more robust to hardware noise.
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('ibm_pittsburgh')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

The QAOA optimization loop runs inside a Qiskit Runtime [session](/docs/guides/execution-modes) to keep the device reserved across iterations. An Estimator evaluates $\langle H_C \rangle$ at each step, and a classical optimizer (COBYLA) updates the parameters until convergence.

![Illustration showing the behavior of Single job, Batch, and Session runtime modes.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Define initial parameters and run the optimization loop:

In [7]:
# QAOA doesn't prescribe principled default angles — any bounded choice
# works as a warm start for problems this small. beta and gamma are
# periodic (beta in [0, pi] and gamma in [0, 2*pi] modulo the underlying
# Pauli-rotation periods), and pi/2 and pi are just midpoints of those
# ranges. For harder problems you would typically warm start from known
# good angles or transfer parameters from smaller instances.
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    estimator.options.environment.job_tags = ["TUT_QAOA"]

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -2.0402211719947774
       x: [ 3.041e+00  1.212e+00  2.081e+00  4.471e+00]
    nfev: 36
   maxcv: 0.0


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### Schritt 3: Ussföhrung met Qiskit Primitives
Em QAOA-Workflow weede de optimal QAOA-Parameter in ener iterative Optimierungsschleif jefunge, wo en Reih vun Schaltkreisbewertunge ussföhrt un ene klassische Optimierer verwend, öm de optimal $\beta_k$ un $\gamma_k$ Parameter ze finge. Die Ussföhrungsschleif weed övver de folgende Schrette ussjeföhrt:

1. Definiere vun de initiale Parameter
2. Instanziierung vun ener neue `Session`, wo de Optimierungsschleif un dat Primitive enthält, wat zom Sample vum Schaltkreis verwend weed
3. Sobald en optimal Parametersatz jefunge es, föhrt mer dä Schaltkreis e letzts Mol us, öm en final Verdeiling ze erhalte, wo em Post-Processing-Schritt verwend weed.
#### Schaltkreis met initiale Parameter definiere
Mer bejinne met willkürlich jewählte Parameter.

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### Backend un Ussföhrungs-Primitive definiere
Verwend de **Qiskit Runtime Primitives**, öm met IBM&reg; Backends ze interagiere. De beid Primitives sin Sampler un Estimator, un de Wahl vum Primitive hänk dovun av, welch Art vun Messung mer opem Quantencomputer ussföhre welle. För de Minimierung vun $H_C$ verwende mer dä Estimator, do de Messung vun dä Kostenfunktion einfach dä Erwartungswert vun $\langle H_C \rangle$ es.
#### Ussföhre
De Primitives biede en Vielzahl vun [Ussföhrungsmodi](/guides/execution-modes) zor Planung vun Workloads op Quantenjeräte, un en QAOA-Workflow löpp iterativ in ener Session.

![Illustration showing the behavior of Single job, Batch, and Session runtime modes.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Mer künne de sampler-baseet Kostenfunktion in de SciPy-Minimierungsroutin eistecke, öm de optimal Parameter ze finge.

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

sampler.options.environment.job_tags = ["TUT_QAOA"]

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{18: 0.039, 5: 0.0665, 20: 0.0973, 29: 0.0063, 9: 0.0899, 13: 0.0379, 2: 0.0047, 1: 0.0153, 11: 0.0932, 14: 0.0327, 12: 0.0314, 25: 0.0193, 21: 0.0398, 6: 0.0224, 4: 0.0197, 10: 0.0387, 3: 0.0181, 26: 0.07, 17: 0.0327, 19: 0.0332, 22: 0.0914, 24: 0.007, 0: 0.0033, 8: 0.0066, 30: 0.0158, 28: 0.0169, 27: 0.0222, 16: 0.0073, 7: 0.0057, 23: 0.0062, 15: 0.0054, 31: 0.0041}


### Step 4: Post-process and return result in desired classical format

Extract the most likely bitstring from the sampled distribution. This represents the best cut found by QAOA.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 0, 1, 0, 1]


In [14]:
plt.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

Sobald mer de optimal Parameter för dä Schaltkreis jefunge han, künne mer die Parameter zowies un de met de optimeet Parameter erhaltene final Verdeiling sample. Hee sollt dat *Sampler* Primitive verwend weede, do et de Wahrscheinlichkeitsverdeiling vun Bitstring-Messunge es, wo däm optimal Schnitt vum Grafen entspreche.

**Pass op:** Dat bedütt, ene Quantenzostand $\psi$ em Computer ze präpariere un en dann ze messe. En Messung weed dä Zostand in ene einzelne Berechnungsbasiszustand kollabeere losse - zom Beispiel `010101110000...` - dä ener Kandidatelösung $x$ för unser ursprüngliche Optimierungsproblem ($\max f(x)$ oder $\min f(x)$ je noh Opjov) entsprich.

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

Now, calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


For a graph this small, the true optimum is easy to brute-force, so you can double-check the results by comparing the QAOA result against the exact answer.

In [17]:
# Classical baseline: enumerate all 2**n_small bitstrings and take the best cut.
def brute_force_max_cut(graph: rx.PyGraph) -> tuple[int, list[int]]:
    n = len(list(graph.nodes()))
    best_cut = -1
    best_x: list[int] = []
    for i in range(2**n):
        x = [(i >> k) & 1 for k in range(n)]
        cut = evaluate_sample(x, graph)
        if cut > best_cut:
            best_cut = int(cut)
            best_x = x
    return best_cut, best_x


classical_best, classical_x = brute_force_max_cut(graph)
print(f"Classical optimum (brute force): {classical_best}")
print(f"QAOA cut value:                  {cut_value}")

Classical optimum (brute force): 5
QAOA cut value:                  5


### Schritt 4: Nohbearbeitung un Röckjov vum Ergebnis em jewünschte klassische Format
Dä Nohbearbeitungsschritt interpreteet de Sampling-Ussjov, öm en Lösung för Eur ursprüngliche Problem zörckzejonn. In däm Fall sin mer an däm Bitstring met dä höchste Wahrscheinlichkeit interesseet, do dä dä optimal Schnitt bestimmp. De Symmetrien em Problem erlaube vier möglich Lösunge, un dä Sampling-Prozess weed ein dovun met ener jet höhere Wahrscheinlichkeit zörckjonn, ävver mer künne in dä unge darjestellte Verdeiling sinn, dat vier vun de Bitstrings deutlich wahrscheinlicher sin als dä Rest.

In [ ]:
# Precomputed parity lookup table: _PARITY[b] = +1 if popcount(b) is even, else -1.
# We use this to vectorize expectation-value evaluation across all Pauli terms.
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Expectation value of a SparsePauliOp on a single computational-basis state.

    For a Z-only observable (which QAOA cost Hamiltonians are, after the
    QUBO-to-Hamiltonian mapping), the eigenvalue of each Pauli term on a
    computational-basis state is simply (-1)**popcount(z_mask AND state),
    i.e., the parity of the bitwise-AND of the term's Z-support and the
    measured bitstring.

    This routine packs the Z-support of every Pauli term into bytes, ANDs
    them against the measured state in a single vectorized op, and looks up
    the parity in _PARITY. For a 100-qubit / ~hundreds-of-terms Hamiltonian
    over 10_000 samples, this is dramatically faster than calling
    SparsePauliOp.expectation_value per sample.
    """
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Return the sampled bitstring (as int) with the lowest Hamiltonian cost."""
    min_cost = float("inf")
    min_sol = None
    for bit_str in samples.keys():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_cost = fval
            min_sol = candidate_sol
    return min_sol


def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(dist, ax, "C1")
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""
    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob
    return objective_values

**Step 1**: Build the graph, cost Hamiltonian, and ansatz.

In [19]:
# Step 1: build the 100-node graph, cost Hamiltonian, and QAOA ansatz.
n_large = 100
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n_large, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n_large and edge[1] < n_large:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)

max_cut_paulis_100 = build_max_cut_paulis(graph_100)
cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(
    max_cut_paulis_100, n_large
)

circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

**Step 2**: Transpile for the selected hardware backend.

In [20]:
# Step 2: transpile for hardware.
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
candidate_circuit_100 = pm.run(circuit_100)

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### Beste Schnitt visualisiere
Us däm optimale Bit-String künne mer dann dä Schnitt opem originale Grafen visualisiere.

In [21]:
# Step 3: run the QAOA optimization loop on the device, then sample the
# final distribution with the optimized parameters.
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    estimator.options.environment.job_tags = ["TUT_QAOA"]

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

# Assign optimal parameters and sample the final distribution.
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)

sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

# Add a unique tag to the job execution
sampler.options.environment.job_tags = ["TUT_QAOA"]

pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -17.172689238986344
       x: [ 2.574e+00  4.166e+00]
    nfev: 28
   maxcv: 0.0


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

Un berechne dä Wert vum Schnitt:

In [22]:
# Step 4: find the best-cost sample and evaluate its cut value.
best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

Result bitstring: [1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0]
The value of the cut is: 156


Check that the cost minimized in the optimization loop has converged, and visualize results.

In [23]:
# Plot convergence
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

# Visualize the cut
plot_result(graph_100, best_sol_bitstring_100)

# Plot cumulative distribution function
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, backend.name)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-2.avif" alt="Output of the previous code cell" />

## Deil II. Opskaleere!
Mer han Zogang zo vill Jeräte met övver 100 Qubits op dä IBM Quantum&reg; Platform. Wähl eins us, op däm mer Max-Cut op enem 100-Knoten jewichtete Grafen löse. Dat es en "utility-scale" Problem. De Schrette för de Workflow ze baue weede genau esu befolch wie ovve, ävver met enem vill jrößere Grafen.